# $N_{\rm H}$ estimate

In this notebook, we compare the estimate of the hydrogen column density $N_{\rm H}$ with two methods:
1. The first one uses a 3D reddening map to estimate the $N_{\rm H}$ value (see [Doroshenko 2024](https://ui.adsabs.harvard.edu/abs/2024arXiv240303127D/abstract)).
2. The second one uses the relation between $N_{\rm H}$ and the dispersion measure DM found by [He, Ng and Kaspi (2013)](https://ui.adsabs.harvard.edu/abs/2013ApJ...768...64H/abstract).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd

import pypopsyn.simulator.interstellar_medium.nh_model as nhm
import utilities.plot_settings

Import an example dataset of detected neutron stars to have a set of neutron stars with sky position in equatorial coordinates and the DM measurement. 

In [ ]:
data_full = pd.read_pickle(
    "../examples/data/simulation_full_edm_example/survey_HTRU_low_mid_results.pkl.gz",
    compression="gzip",
)

In [ ]:
RA = data_full["RA"]["[deg]"].to_numpy()
DEC = data_full["DEC"]["[deg]"].to_numpy()
d = data_full["d"]["[kpc]"].to_numpy()
DM = data_full["DM"]["[pc cm^-3]"].to_numpy()

Estimate the $N_{\rm H}$ with the two methods and plot the results to compare them. As you can note, despite the large scatter, the $N_{\rm H}$ estimated from the $N_{\rm H}$ - DM relation tends to underestimate the $N_{\rm H}$ value at high $N_{\rm H}$ values.

In [ ]:
N_H = nhm.compute_NH(RA, DEC, d)
N_H_dm = nhm.compute_NH_from_DM(DM)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.scatter(
    N_H,
    N_H_dm,
    s=10,
    color="black",
    alpha=1,
)
ax.plot(
    np.logspace(21, 23, 1000),
    np.logspace(21, 23, 1000),
    linestyle="--",
    lw=4,
    color="tab:red",
    alpha=1,
)
plt.xlabel(r"$N_{\rm H}$ 3D-tool")
plt.ylabel(r"$N_{\rm H}$ from DM")
plt.xscale("log")
plt.yscale("log")

Load the reddening map to visualize it. The colorcode is in logarithmic scale as it helps to show better the structures in the map. As you move from left to right the heliocentric distance increases and from top to bottom you are looking at various line of sights in the sky. The value in each bin of the map should be the total reddening E(B-V) in mag for a given distance and line of sight.

In [ ]:
maps = np.load("../pypopsyn/simulator/interstellar_medium/ebv_map.npz")

In [ ]:
print(maps["maps"].shape)
print(maps["radius"].shape)
print(np.min(maps["maps"]))
print(np.max(maps["maps"]))
print(np.min(maps["radius"]))
print(np.max(maps["radius"]))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 12))

img = ax.imshow(
    np.log10(maps["maps"].T),
    cmap='viridis', 
    aspect='auto'
)
plt.xlabel('Heliocentric distance bins')
plt.ylabel('Sky position bins')
plt.colorbar(img, label=r'$\log_{10} \left( {\rm E(B-V) \, [mag]} \right)$')